# Import Dependencies

In [1]:
import torch
from urllib.request import urlopen
from PIL import Image
from open_clip import create_model_from_pretrained, get_tokenizer

# Load BiomedCLIP

In [16]:

# Load the model and config files from the Hugging Face Hub
model, preprocess = create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f'Using device: {device}')
model.to(device)
model.eval()


Using device: cuda


CustomTextCLIP(
  (visual): TimmModel(
    (trunk): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=768

# Set up labels and list of imaged from DIBaS_Dataset

In [18]:
# Zero-shot image classification
template = 'this is a photo of '
labels = [
"Acinetobacter",
"Actinomyces",
"Bacteroides",
"Bifidobacterium",
"Candida",
"Clostridium",
"Enterococcus",
"Escherichia.coli",
"Fusobacterium",
"Lactobacillus",
"Listeria",
"Micrococcus",
"Neisseria",
"Porfyromonas",
"Propionibacterium",
"Proteus",
"Pseudomonas.aerunosa",
"Staphylococcus",
"Streptococcus",
"Veionella"]

test_imgs = [
    '../data/DIBaS_Dataset/Acinetobacter.baumanii/Acinetobacter.baumanii_0001.tif',
    '../data/DIBaS_Dataset/Actinomyces.israeli/Actinomyces.israeli_0001.tif',
    '../data/DIBaS_Dataset/Veionella/Veionella_0001.tif',
    '../data/DIBaS_Dataset/Streptococcus.agalactiae/Streptococcus.agalactiae_0001.tif'
]


# Run inference to classify images

In [19]:

context_length = 256

images = torch.stack([preprocess(Image.open(img)) for img in test_imgs]).to(device)
texts = tokenizer([template + l for l in labels], context_length=context_length).to(device)
with torch.no_grad():
    image_features, text_features, logit_scale = model(images, texts)

    logits = (logit_scale * image_features @ text_features.t()).detach().softmax(dim=-1)
    sorted_indices = torch.argsort(logits, dim=-1, descending=True)

    logits = logits.cpu().numpy()
    sorted_indices = sorted_indices.cpu().numpy()

top_k = -1

for i, img in enumerate(test_imgs):
    pred = labels[sorted_indices[i][0]]

    top_k = len(labels) if top_k == -1 else top_k
    print(img.split('/')[-1] + ':')
    for j in range(top_k):
        jth_index = sorted_indices[i][j]
        print(f'{labels[jth_index]}: {logits[i][jth_index]}')
    print('\n')

Acinetobacter.baumanii_0001.tif:
Fusobacterium: 0.5770657658576965
Lactobacillus: 0.2800767123699188
Escherichia.coli: 0.09819173067808151
Listeria: 0.014799477532505989
Bacteroides: 0.014388096518814564
Micrococcus: 0.011324450373649597
Staphylococcus: 0.002262821653857827
Acinetobacter: 0.0009966033976525068
Enterococcus: 0.00039010882028378546
Clostridium: 0.00016434650751762092
Neisseria: 0.00015756439825054258
Pseudomonas.aerunosa: 0.00012138045713072643
Streptococcus: 3.7556903407676145e-05
Veionella: 1.2265528312127572e-05
Candida: 1.0294427738699596e-05
Bifidobacterium: 7.068920240271837e-07
Propionibacterium: 1.199400827545105e-07
Actinomyces: 6.526151175023642e-09
Porfyromonas: 9.913786458426443e-11
Proteus: 2.4786507481916464e-12


Actinomyces.israeli_0001.tif:
Lactobacillus: 0.4516614079475403
Listeria: 0.3609568178653717
Streptococcus: 0.05950215086340904
Neisseria: 0.05087101086974144
Escherichia.coli: 0.026296967640519142
Fusobacterium: 0.024070221930742264
Staphylococcu

# YEP - No Bueno